In [ ]:
import re
import numpy as np
import sklearn as sns
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_excel('Online Retail.xlsx')

In [ ]:
print("\n--- TỔNG QUAN DỮ LIỆU ---")
print(f"Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")
print(df.info())

In [ ]:
print("\n--- KIỂM TRA GIÁ TRỊ RỖNG ---")
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Số lượng rỗng': missing_data, 'Tỷ lệ (%)': missing_percent})
print(missing_data)
print(missing_df[missing_df['Số lượng rỗng'] > 0])

In [ ]:
# 4. Phân tích bất thường (Thống kê mô tả)
print("\n--- THỐNG KÊ MÔ TẢ CÁC CỘT SỐ ---")
display(df.describe(include='all'))

In [ ]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
cancelled_orders = df[df['InvoiceNo'].str.startswith('C')]
print(f"\nSố lượng giao dịch bị hủy (Invoice bắt đầu bằng 'C'): {len(cancelled_orders)}")
print(f"Số lượng giao dịch có Quantity âm: {len(df[df['Quantity'] <= 0])}")

In [ ]:
print("\n--- ĐẶC TRƯNG CHUỖI KHÁCH HÀNG ---")
print(f"Thời gian bắt đầu: {df['InvoiceDate'].min()}")
print(f"Thời gian kết thúc: {df['InvoiceDate'].max()}")
print(f"Số lượng khách hàng duy nhất: {df['CustomerID'].nunique()}")
print(f"Số lượng mặt hàng (StockCode) duy nhất: {df['StockCode'].nunique()}")

In [ ]:
temp_df = df.dropna(subset=['CustomerID'])
visits_per_customer = temp_df.groupby('CustomerID')['InvoiceNo'].nunique()

print(f"\nTrung bình một khách có bao nhiêu hóa đơn (lần ghé thăm): {visits_per_customer.mean():.2f}")
print(f"Khách hàng mua nhiều lần nhất: {visits_per_customer.max()} lần")

In [ ]:
print("\n--- BƯỚC 1: LÀM SẠCH DỮ LIỆU ---")
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
df_clean = df_clean[df_clean['StockCode'].astype(str).str.contains(r'\d', regex=True, na=False)]
df_clean = df_clean.drop_duplicates()
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f"-> Số dòng gốc: {len(df)}")
print(f"-> Số dòng sau khi làm sạch (df_clean): {len(df_clean)}")

In [ ]:
print("\n--- BƯỚC 2: TẠO CHUỖI TUẦN TỰ (SEQUENCES) ---")

df_clean = df_clean.sort_values(by=['CustomerID', 'InvoiceDate'])
df_clean['StockCode'] = df_clean['StockCode'].astype(str).str.strip()

print("-> Đang gom nhóm các sản phẩm cùng 1 hóa đơn...")

itemsets = df_clean.groupby(['CustomerID', 'InvoiceNo', 'InvoiceDate'])['StockCode'].apply(lambda x: sorted(list(set(x)))).reset_index()

print("\n--- KẾT QUẢ TẠO GIỎ HÀNG (ITEMSETS)---")
display(itemsets.head())

print("\n-> Đang nối các giỏ hàng thành chuỗi hành vi của từng khách hàng...")
sequences = itemsets.groupby('CustomerID')['StockCode'].apply(list).reset_index()

print("\n--- KẾT QUẢ TẠO CHUỖI TUẦN TỰ (SEQUENCES) ---")
display(sequences.head())
output_filename = 'prefixspan_online_retail.txt'

with open(output_filename, 'w', encoding='utf-8') as f:
    for seq in sequences['StockCode']:
        seq_str = ""
        for itemset in seq:
            itemset_str = " ".join(itemset)
            seq_str += f"{itemset_str} -1 "

        seq_str += "-2\n"
        f.write(seq_str)

print(f"Xử lý hoàn tất. Tập dữ liệu gồm {len(sequences)} chuỗi tuần tự đã được lưu tại: {output_filename}")


In [ ]:
sns.set_theme(style="whitegrid")
sequence_lengths = sequences['StockCode'].apply(len)
plt.figure(figsize=(10, 6))
# Giới hạn xem các khách hàng mua dưới 15 lần để biểu đồ không bị nhiễu bởi các outlier
sns.histplot(sequence_lengths[sequence_lengths <= 15], bins=15, color='royalblue', kde=True)

plt.title('Biểu đồ 1: Phân bố số lần mua sắm của khách hàng (Độ dài chuỗi)', fontsize=14, fontweight='bold')
plt.xlabel('Số lần mua hàng (Sequence Length)', fontsize=12)
plt.ylabel('Số lượng khách hàng', fontsize=12)
plt.xticks(range(1, 16))
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
all_items = []
for seq in sequences['StockCode']:
    for itemset in seq:
        all_items.extend(itemset)

item_counts = Counter(all_items)
top_10_items = item_counts.most_common(10)
table2_data = {
    'Mã sản phẩm (StockCode)': [x[0] for x in top_10_items],
    'Tần suất xuất hiện (Support count)': [x[1] for x in top_10_items]
}
df_top10 = pd.DataFrame(table2_data)
plt.figure(figsize=(12, 6))
sns.barplot(data=df_top10, x='Mã sản phẩm (StockCode)', y='Tần suất xuất hiện (Support count)', palette='viridis')

plt.title('Biểu đồ 2: Top 10 sản phẩm được mua nhiều nhất', fontsize=14, fontweight='bold')
plt.xlabel('Mã sản phẩm', fontsize=12)
plt.ylabel('Tần suất (Số lần xuất hiện)', fontsize=12)
# Xoay nhãn trục x nếu cần
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

from collections import Counter
itemset_sizes = itemsets['StockCode'].apply(len)
size_counts = itemset_sizes.value_counts().sort_index().head(10)

table_size_data = {
    'Số lượng sản phẩm / 1 lần mua (Itemset Size)': size_counts.index,
    'Số lượng giỏ hàng (Tần suất)': size_counts.values,
    'Tỷ lệ (%)': np.round((size_counts.values / len(itemsets)) * 100, 2)
}
df_itemset_size = pd.DataFrame(table_size_data)

print("\nBẢNG 3: PHÂN BỐ KÍCH THƯỚC GIỎ HÀNG TRONG 1 LẦN GIAO DỊCH")
print("-" * 65)
print(df_itemset_size.to_string(index=False))

In [ ]:
transitions = []

for seq in sequences['StockCode']:
    if len(seq) > 1:
        for i in range(len(seq) - 1):
            itemset_current = seq[i]
            itemset_next = seq[i+1]
            for item1 in itemset_current:
                for item2 in itemset_next:
                    transitions.append(f"{item1} -> {item2}")

top_transitions = Counter(transitions).most_common(10)

table_transition_data = {
    'Mẫu chuỗi 2 phần tử (Transition)': [x[0] for x in top_transitions],
    'Số lần xuất hiện': [x[1] for x in top_transitions]
}
df_transitions = pd.DataFrame(table_transition_data)
print("\nBẢNG 4: TOP 10 CẶP SẢN PHẨM ĐƯỢC MUA LIÊN TIẾP NHAU (CHUYỂN ĐỔI 1 BƯỚC)")
print("-" * 65)
print(df_transitions.to_string(index=False))

In [ ]:
# ==========================================
# TRIỂN KHAI THUẬT TOÁN PREFIXSPAN TỪ ĐẦU
# ==========================================

class CustomPrefixSpan:
    def __init__(self, sequences, min_support):
        self.sequences = sequences
        self.min_support = min_support
        self.frequent_patterns = []

    def mine(self):
        """Hàm kích hoạt thuật toán"""
        self.frequent_patterns = []
        # Khởi chạy đệ quy với CSDL ban đầu (projected_db) và tiền tố rỗng (prefix=[])
        self._mine_recursive(self.sequences, prefix=[])
        return self.frequent_patterns

    def _mine_recursive(self, projected_db, prefix):
        """VÒNG LẶP ĐỆ QUY CỐT LÕI (Pattern-growth)"""
        
        # Bước 1: Đếm tần suất (support count) của các item trong CSDL chiếu hiện tại
        item_counts = {}
        for sequence in projected_db:
            # Dùng set() để đảm bảo mỗi sequence chỉ đóng góp 1 support cho 1 item
            unique_items_in_seq = set()
            for itemset in sequence:
                for item in itemset:
                    unique_items_in_seq.add(item)
            
            for item in unique_items_in_seq:
                item_counts[item] = item_counts.get(item, 0) + 1

        # Bước 2: Lọc các item phổ biến (thỏa mãn ngưỡng min_support)
        frequent_items = {item: count for item, count in item_counts.items() if count >= self.min_support}

        # Bước 3: Phát triển tiền tố (Prefix) và gọi đệ quy
        for item, count in frequent_items.items():
            # Tạo mẫu mới bằng cách nối item phổ biến vào tiền tố hiện tại
            new_prefix = prefix + [item]
            self.frequent_patterns.append((new_prefix, count))

            # Xây dựng Cơ sở dữ liệu chiếu (Projected Database) tương ứng với mẫu mới
            new_projected_db = self._build_projected_db(projected_db, item)

            # ĐIỀU KIỆN DỪNG ĐỆ QUY: 
            # Nếu CSDL chiếu rỗng (tức là không còn item nào xuất hiện sau tiền tố này), đệ quy tự động dừng.
            # Nếu không rỗng, tiếp tục khai phá trên CSDL chiếu đã thu nhỏ.
            if new_projected_db:
                self._mine_recursive(new_projected_db, new_prefix)

    def _build_projected_db(self, projected_db, item):
        """Hàm cắt CSDL ban đầu thành CSDL chiếu dựa trên tiền tố"""
        new_db = []
        for sequence in projected_db:
            for i, itemset in enumerate(sequence):
                # Tìm thấy vị trí xuất hiện đầu tiên của item trong chuỗi
                if item in itemset:
                    # Cắt bỏ toàn bộ phần trước và giữ lại phần dữ liệu SAU vị trí xuất hiện
                    # Lưu ý học thuật: Bản cài đặt này tập trung tìm chuỗi tuần tự theo chiều dọc thời gian (Event A -> Event B)
                    projected_sequence = sequence[i+1:]
                    
                    if projected_sequence:
                        new_db.append(projected_sequence)
                    break # Cắt xong thì thoát vòng lặp, chuyển sang sequence tiếp theo
        return new_db



In [ ]:
import time


print("--- BẮT ĐẦU CHẠY PREFIXSPAN TRÊN DỮ LIỆU THẬT ---")

# Lấy danh sách các chuỗi từ DataFrame 'sequences' đã tạo ở bước EDA
# Mỗi phần tử trong list này là 1 chuỗi hành vi của 1 khách hàng
real_sequences = sequences['StockCode'].tolist()

# Đặt ngưỡng min_support (Ví dụ: 2% trên tổng số khách hàng)
# Tập dữ liệu có 4334 khách hàng, 2% tương đương khoảng 86
min_sup_ratio = 0.02
min_sup_count = int(len(real_sequences) * min_sup_ratio)

print(f"Tổng số chuỗi (khách hàng): {len(real_sequences)}")
print(f"Ngưỡng Min Support: {min_sup_ratio*100}% (Yêu cầu xuất hiện ít nhất {min_sup_count} lần)")
print("Đang tiến hành khai phá mẫu... (Vui lòng đợi, có thể mất một chút thời gian)")

# Bắt đầu đo thời gian chạy (phục vụ cho báo cáo Chương 4)
start_time = time.time()

# Khởi tạo và kích hoạt thuật toán
ps_real = CustomPrefixSpan(real_sequences, min_support=min_sup_count)
real_patterns = ps_real.mine()

# Kết thúc đo thời gian
end_time = time.time()
execution_time = end_time - start_time

print(f"\n--- KẾT QUẢ KHAI PHÁ ---")
print(f"Thời gian chạy thuật toán: {execution_time:.2f} giây")
print(f"Tổng số lượng mẫu tuần tự phổ biến tìm được: {len(real_patterns)}")

print("\n--- TOP 15 MẪU TUẦN TỰ XUẤT HIỆN NHIỀU NHẤT ---")
# Sắp xếp các mẫu tìm được theo thứ tự Support giảm dần
real_patterns_sorted = sorted(real_patterns, key=lambda x: x[1], reverse=True)

for i, (pattern, support) in enumerate(real_patterns_sorted[:15], 1):
    # Nối các mã sản phẩm bằng mũi tên để dễ nhìn (vd: 22423 -> 22699)
    pattern_str = ' -> '.join(pattern)
    print(f"Top {i}: [ {pattern_str} ]  |  Tần suất (Support): {support}")

### Đánh giá và so sánh kết quả với các ngưỡng Min Support khác nhau

In [ ]:
import time
import pandas as pd
from IPython.display import display, Markdown

min_sup_ratios = [0.05, 0.04, 0.03, 0.02, 0.015, 0.01]
results = []
real_sequences = sequences['StockCode'].tolist()
total_seqs = len(real_sequences)

for ratio in min_sup_ratios:
    min_sup_count = int(total_seqs * ratio)
    start_time = time.time()
    ps = CustomPrefixSpan(real_sequences, min_support=min_sup_count)
    patterns = ps.mine()
    execution_time = time.time() - start_time
    
    results.append({
        'Min Support Ratio': f"{ratio*100}%",
        'Min Support Count': min_sup_count,
        'Số lượng mẫu tìm được': len(patterns),
        'Thời gian chạy (s)': round(execution_time, 4)
    })

df_results = pd.DataFrame(results)
display(df_results)
